# Effect of Antisemitic Tropes on Recall 

In [1]:
import pandas as pd
import os
import sys
from os.path import join, abspath

parent_dir = abspath(os.path.join(os.getcwd(), '../..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from config import DATA_DIR
from utils.classification_helpers import group_ihra_content, group_lexicon_content
from utils.stats_helpers import pairwise_segment_tests
from utils.analysis_helpers import calculate_recall, summarize_recall_stats, group_dfs_by_row_index_pivot, explode_df, test_recall_per_content_group_compared_to_base, combine_dataset_tests

pd.set_option("display.max_rows", 100)

PROVIDER = "llama"

In [2]:
print("test")

test


In [3]:
bloomington_tmp = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
decoding = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))

In [4]:
print("Bloomington columns:", bloomington_tmp.columns)

Bloomington columns: Index(['comment_cleaned', 'id', 'keyword', 'ihra_section_1', 'ihra_section_2',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_tax',
       'explanation_tax', 'classification_tax_ex', 'explanation_tax_ex',
       'classification_no_kb_cleaned', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters', 'explanation_ihra_explanation_sections',
       'ihra_sections', 'explanation_tax_chapters_no',
       'explanation_tax_ex_chapters_no', 'explanation_tax_sections',
       'explanation_tax_ex_sections'],
      dtype='object')


In [5]:
column_name_renaming = {
    'classification_no_kb_cleaned': 'NO_KB',
    'classification_ihra_explanation_cleaned': 'IHRA',
    'classification_tax': 'TAX',
    'classification_tax_ex': 'TAX_EX',
}

bloomington_tmp.rename(columns=column_name_renaming, inplace=True)
decoding.rename(columns=column_name_renaming, inplace=True)

REL_CLASS_COLS = column_name_renaming.values()

In [6]:
# Create a new df for Bloomington data with a new column "ihra_sec_new" containing both annotators' decisions.
bloomington_copy_x = bloomington_tmp.copy()
bloomington_copy_y = bloomington_tmp.copy()
bloomington_copy_x["ihra_sec_new"] = bloomington_tmp["ihra_section_1"]
bloomington_copy_y["ihra_sec_new"] = bloomington_tmp["ihra_section_2"]
bloomington = pd.concat([bloomington_copy_x, bloomington_copy_y], ignore_index=True)
print(len(bloomington))
bloomington = bloomington[bloomington["ihra_sec_new"]!=13] # there are a few errors coded as 13
print(len(bloomington))

3716
3635


## Bloomington

#### a. Within each KB

In [7]:

class_cols_to_recalls = {}
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='keyword', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"------{classification_column}-------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    


------NO_KB-------
         Recall  Support  Correct  Missed
Israel     0.20      641      130     511
Kikes      0.92       89       82       7
ZioNazi    0.86      397      342      55
Jews       0.47      689      320     369

[Pairwise tests vs each other] (Holm-corrected)
     seg_A    seg_B         p_raw         p_adj  significant  effect_h
1   Israel  ZioNazi  1.042323e-94  6.253937e-94         True -1.447303
0   Israel    Kikes  9.875577e-44  4.937788e-43         True -1.640784
5  ZioNazi     Jews  8.470860e-38  3.388344e-37         True  0.863838
2   Israel     Jews  1.264001e-23  3.792002e-23         True -0.583465
4    Kikes     Jews  1.197880e-15  2.395760e-15         True  1.057319
3    Kikes  ZioNazi  1.754697e-01  1.754697e-01        False  0.193481
---------

------IHRA-------
         Recall  Support  Correct  Missed
Israel     0.44      641      281     360
Kikes      1.00       89       89       0
ZioNazi    0.98      397      389       8
Jews       0.68      689    

#### b. Between KBs

In [8]:
grouped = group_dfs_by_row_index_pivot(class_cols_to_recalls)
print(grouped["Kikes"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA          89       0    1.00       89
NO_KB         82       7    0.92       89
TAX           88       1    0.99       89
TAX_EX        88       1    0.99       89


In [9]:
print(grouped["Israel"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA         281     360    0.44      641
NO_KB        130     511    0.20      641
TAX          382     259    0.60      641
TAX_EX       320     321    0.50      641


In [10]:
# Pairwise chi-squared (auto-fallback to Fisher), Holm-corrected
model_comparison_per_group = pd.DataFrame(columns=["seg_A","seg_B", "p_raw","p_adj","significant", "effect_h"])

index = []
for i, k in enumerate(grouped.keys()):
    pairwise = pairwise_segment_tests(grouped[k], correction="holm")
    model_comparison_per_group.loc[i] = pairwise.loc[1] # change 1 to another number to get a different comparison
    index.append(k)
model_comparison_per_group["keyword"] = index
model_comparison_per_group.set_index("keyword", inplace=True)
model_comparison_per_group

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
keyword,,,,,,
Israel,IHRA,TAX,0.0,0.0,True,-0.321648
Jews,IHRA,TAX,0.560711,0.607143,False,-0.043249
Kikes,IHRA,TAX,1.0,1.0,False,0.200315
ZioNazi,IHRA,TAX,0.496864,1.0,False,0.064372


### Evaluation by content groups

Content groups allow to compare IHRA sections with LEXICON chapters

In [11]:
bloomington["ihra_content"] = bloomington["ihra_sec_new"].map(group_ihra_content)
print(bloomington["ihra_content"].value_counts(dropna=False, normalize=True))
print(type(bloomington["ihra_content"][0]))

ihra_content
israel                  0.576891
classic_power           0.279230
aggressive              0.096286
second_postholocaust    0.047593
Name: proportion, dtype: float64
<class 'str'>


#### a. Within each KB

In [12]:
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='ihra_content', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.38     1048      398     650
classic_power           0.62      507      314     193
aggressive              0.74      175      129      46
second_postholocaust    0.40       86       34      52
           seg_A                 seg_B         p_raw         p_adj  \
0         israel         classic_power  1.011981e-18  6.071885e-18   
1         israel            aggressive  2.042410e-18  1.021205e-17   
5     aggressive  second_postholocaust  1.752922e-07  7.011687e-07   
4  classic_power  second_postholocaust  1.555353e-04  4.666058e-04   
3  classic_power            aggressive  6.437616e-03  1.287523e-02   
2         israel  second_postholocaust  8.646270e-01  8.646270e-01   

   significant  effect_h  
0         True -0.484732  
1         True -0.743021  
5         True  0.702013  
4         True  0.443724  
3         True -0.258289  
2        False -0.041008  
---------

--------IHRA

In [13]:
kb_to_recall_b = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_b[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_b)

,NO_KB,IHRA,TAX,TAX_EX
israel,0.38,0.59,0.69,0.62
classic_power,0.62,0.80,0.80,0.77
aggressive,0.74,0.86,0.87,0.87
second_postholocaust,0.40,0.64,0.66,0.62


#### b. Between KBs

In [17]:
recall_tests_b = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,3,4]])
recall_tests_b

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.007849,0.031398,True,-0.303148
classic_power,NO_KB,IHRA,0.0,0.0,True,-0.401135
israel,NO_KB,IHRA,0.0,0.0,True,-0.423352
second_postholocaust,NO_KB,IHRA,0.002275,0.011373,True,-0.485152
aggressive,NO_KB,TAX,0.003118,0.01871,True,-0.332416
classic_power,NO_KB,TAX,0.0,0.0,True,-0.401135
israel,NO_KB,TAX,0.0,0.0,True,-0.632162
second_postholocaust,NO_KB,TAX,0.000778,0.004666,True,-0.527087
aggressive,NO_KB,TAX_EX,0.005012,0.025061,True,-0.332416


## Decoding

### Evaluation by content groups



In [18]:
decoding["lexicon_chapter_groups"] = decoding["comment_codes_all_chapters"].map(group_lexicon_content)

In [19]:
# restrict to texts with maximum two content groups assigned
decoding = decoding[decoding['lexicon_chapter_groups'].map(lambda x: len(x) <=2 and len(x)>0)].copy()
decoding["lexicon_chapter_groups"] = decoding["lexicon_chapter_groups"].map(lambda x: x if len(x)==2 else x*2)
decoding = explode_df(decoding, "lexicon_chapter_groups")

In [20]:
decoding["lexicon_chapter_groups"].value_counts()

lexicon_chapter_groups
israel                  2667
classic_power           2169
second_postholocaust     775
aggressive               131
Name: count, dtype: int64

#### a. Within each KB

In [21]:
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=decoding, classification_column=classification_column, split_by='lexicon_chapter_groups', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")   

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.27     1333      355     978
classic_power           0.32     1084      346     738
second_postholocaust    0.18      387       69     318
aggressive              0.71       65       46      19

[Pairwise tests vs each other] (Holm-corrected)
                  seg_A                 seg_B         p_raw         p_adj  \
5  second_postholocaust            aggressive  4.928422e-19  2.957053e-18   
2                israel            aggressive  4.618274e-14  2.309137e-13   
4         classic_power            aggressive  3.334221e-10  1.333688e-09   
3         classic_power  second_postholocaust  1.777295e-07  5.331884e-07   
1                israel  second_postholocaust  5.203504e-04  1.040701e-03   
0                israel         classic_power  5.050309e-03  5.050309e-03   

   significant  effect_h  
5         True -1.127944  
2         True -0.911441  
4         True -0.801713  
3     

In [22]:
kb_to_recall_d = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_d[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_d)

,NO_KB,IHRA,TAX,TAX_EX
israel,0.27,0.57,0.69,0.62
classic_power,0.32,0.58,0.68,0.61
second_postholocaust,0.18,0.36,0.46,0.40
aggressive,0.71,0.79,0.87,0.84


#### b. Between KBs

In [26]:
recall_tests_d = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,3,4]])
recall_tests_d

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.308671,1.0,False,-0.185283
classic_power,NO_KB,IHRA,0.0,0.0,True,-0.528959
israel,NO_KB,IHRA,0.0,0.0,True,-0.618457
second_postholocaust,NO_KB,IHRA,0.0,0.0,True,-0.410704
aggressive,NO_KB,TAX,0.030612,0.183673,False,-0.399625
classic_power,NO_KB,TAX,0.0,0.0,True,-0.736536
israel,NO_KB,TAX,0.0,0.0,True,-0.867791
second_postholocaust,NO_KB,TAX,0.0,0.0,True,-0.614413
aggressive,NO_KB,TAX_EX,0.091912,0.459561,False,-0.314317


## Dataset union 

- equal dataset weight
- by content groups

In [27]:
# combine recalls per content group
# also need to fetch overall recall first

In [28]:
kb_to_recall_bd = {}
for classification_column in REL_CLASS_COLS:
    tmp_b = kb_to_recall_b[classification_column]
    tmp_d = kb_to_recall_d[classification_column]
    tmp = {k:0.5*(tmp_b[k] + tmp_d[k]) for k in tmp_b.keys()}
    kb_to_recall_bd[classification_column] = tmp
       

In [29]:
kb_to_recall_bd = pd.DataFrame(kb_to_recall_bd)
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX
israel,0.325,0.580,0.69,0.620
classic_power,0.470,0.690,0.74,0.690
aggressive,0.725,0.825,0.87,0.855
second_postholocaust,0.290,0.500,0.56,0.510


In [30]:
kb_to_recall_bd.loc["mean",:] = kb_to_recall_bd.mean().tolist()
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX
israel,0.3250,0.58000,0.690,0.62000
classic_power,0.4700,0.69000,0.740,0.69000
aggressive,0.7250,0.82500,0.870,0.85500
second_postholocaust,0.2900,0.50000,0.560,0.51000
mean,0.4525,0.64875,0.715,0.66875


In [31]:
kb_to_recall_bd.to_parquet(f"{PROVIDER}_recall_by_trope.parquet", index=True)


In [32]:
combined = combine_dataset_tests(
    recall_tests_b,
    recall_tests_d,
)
combined

,content_group,seg_A,seg_B,p_raw,effect_h,p_adj,significant
2,israel,NO_KB,IHRA,7.274069e-30,-0.520904,8.728883e-29,True
6,israel,NO_KB,TAX,7.274069e-30,-0.749977,8.728883e-29,True
10,israel,NO_KB,TAX_EX,7.274069e-30,-0.602546,8.728883e-29,True
1,classic_power,NO_KB,IHRA,4.092051e-24,-0.465047,3.682846e-23,True
5,classic_power,NO_KB,TAX,4.092051e-24,-0.568836,3.682846e-23,True
9,classic_power,NO_KB,TAX_EX,5.781248e-21,-0.459077,4.046874e-20,True
7,second_postholocaust,NO_KB,TAX,8.132751e-16,-0.570750,4.879650e-15,True
11,second_postholocaust,NO_KB,TAX_EX,3.963580e-11,-0.468432,1.981790e-10,True
3,second_postholocaust,NO_KB,IHRA,7.034585e-10,-0.447928,2.813834e-09,True
4,aggressive,NO_KB,TAX,2.958526e-04,-0.366020,8.875579e-04,True
